# Panel de jueces LLM v2 (A2)

Segunda versión del panel con los mismos tres jueces (Llama 3.1 8B, Qwen 2.5 7B y Mistral 7B v0.3) servidos con vLLM en una GPU A100. Cambios frente a la v1, según la auditoría del panel:

- El prompt solo usa el contexto de la cita, sin la sección retórica (decisión 5).
- La justificación de cada juez se recorta a 300 caracteres para que la salida estructurada no falle (decisión 6).
- train, dev y test conservan el orden de `data/dataset` (decisión 3).

La v1 no se toca: sus resultados siguen en `MyDrive/EvidenCite_A2/labeled` y los de esta versión quedan en `MyDrive/EvidenCite_A2/labeled_v2`.

Antes de ejecutar:

- Subir `EvidenCite_A2_v2.zip` a `MyDrive/EvidenCite_A2/`.
- Tener el secreto `HF_TOKEN` en Colab con acceso a `meta-llama/Llama-3.1-8B-Instruct` y `mistralai/Mistral-7B-Instruct-v0.3`.
- Entorno de ejecución con GPU A100.

El notebook corre las 18.658 citas (`SAMPLE = None`). Para una prueba corta se puede poner `SAMPLE = 60`, pero la comparación con la v1 de la última celda solo tiene sentido con la corrida completa. Los votos quedan en caché en Drive, así que si la sesión se cae se puede retomar.

Librerías necesarias: vllm, openai, pydantic, pyyaml, scikit-learn.


In [ ]:
!pip install -q vllm openai pydantic pyyaml scikit-learn

In [ ]:
import json, os, shutil, signal, subprocess, time, urllib.request, zipfile
from importlib.metadata import version
from pathlib import Path
import yaml
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/EvidenCite_A2')
REPO = Path('/content/EvidenCite_v2')
if not REPO.exists():
    with zipfile.ZipFile(DRIVE / 'EvidenCite_A2_v2.zip') as z:
        z.extractall(REPO)
os.chdir(REPO)
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
for name in ['labeled_v2', 'logs']:
    if (DRIVE / name).exists():
        shutil.copytree(DRIVE / name, REPO / 'data' / name, dirs_exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('vllm', version('vllm'))

In [ ]:
CONFIG = 'configs/labeling_v2.yaml'
JUDGES = {j['id']: j['model_name'] for j in yaml.safe_load(open(CONFIG))['judges']}
SAMPLE = None
JUDGES

In [ ]:
def start_server(model):
    log = open(f"/content/vllm_{model.split('/')[-1]}.log", 'w')
    proc = subprocess.Popen(
        ['vllm', 'serve', model, '--port', '8000', '--max-model-len', '4096', '--gpu-memory-utilization', '0.90'],
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
    )
    t0 = time.time()
    while time.time() - t0 < 1800:
        if proc.poll() is not None:
            raise RuntimeError(f'vLLM se detuvo, revisar {log.name}')
        try:
            with urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5) as r:
                if r.status == 200:
                    print(model, f'listo en {time.time() - t0:.0f}s')
                    return proc
        except Exception:
            pass
        time.sleep(10)
    stop_server(proc)
    raise TimeoutError(model)


def stop_server(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(timeout=120)
    except ProcessLookupError:
        pass
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
    time.sleep(15)


def save_to_drive():
    for name in ['labeled_v2', 'logs']:
        shutil.copytree(REPO / 'data' / name, DRIVE / name, dirs_exist_ok=True)


def run_panel(*extra):
    args = ['python', '-m', 'src.labeling.judge_panel', '--config', CONFIG, *extra]
    if SAMPLE:
        args += ['--sample', str(SAMPLE)]
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end='')
    if p.returncode:
        raise RuntimeError(f'judge_panel termino con codigo {p.returncode}')

In [ ]:
for judge_id, model in JUDGES.items():
    t0 = time.time()
    proc = start_server(model)
    try:
        run_panel('--judges', judge_id)
    finally:
        stop_server(proc)
        save_to_drive()
    print(judge_id, f'{(time.time() - t0) / 60:.1f} min')

In [ ]:
run_panel('--cache-only')
save_to_drive()

In [ ]:
m = json.loads((REPO / 'data' / 'labeled_v2' / 'labeling_metrics.json').read_text())
for k in ['total_instances', 'splits', 'label_source_distribution', 'decision_cases', 'pairwise_cohen_kappa', 'mean_pairwise_kappa']:
    print(k, m[k])
g = m['gold_calibration']
print('gold', g['evaluated_instances'], g['panel_accuracy'], g['per_judge_accuracy'])
for j, u in m['judge_usage'].items():
    print(j, 'votos', u['votes'], 'abstenciones', u['abstentions'], 'latencia_ms', round(u['mean_latency_ms']))

In [ ]:
import pandas as pd

v1 = json.loads((DRIVE / 'labeled' / 'labeling_metrics.json').read_text())
v2 = m


def r(valor, n=3):
    return None if valor is None else round(valor, n)


def resumen(x):
    fila = {
        'instancias': x['total_instances'],
        'llm_panel': x['label_source_distribution'].get('llm_panel', 0),
        'needs_review': x['label_source_distribution'].get('needs_review', 0),
        'kappa_medio': r(x['mean_pairwise_kappa']),
        'exactitud_gold_panel': r(x['gold_calibration']['panel_accuracy']),
    }
    for caso in 'ABCDEF':
        fila[f'caso_{caso}'] = x['decision_cases'].get(caso, 0)
    for j, acc in x['gold_calibration']['per_judge_accuracy'].items():
        fila[f'exactitud_gold_{j}'] = r(acc)
    for j, u in x['judge_usage'].items():
        fila[f'abstenciones_{j}'] = u['abstentions']
        fila[f'latencia_ms_{j}'] = r(u['mean_latency_ms'], 0)
    fila['clases_bajo_cuota'] = sum(1 for q in x['quota_status'].values() if q['deficit'] > 0)
    return fila


pd.DataFrame({'v1': resumen(v1), 'v2': resumen(v2)})